# Bronze to Silver

Tratamento das tabelas `v_energy.bronze` para `v_energy.silver`, uma seção por tabela. Cada decisão de limpeza
tem um ID (`UC-01`, `FAT-02`...) registrado em `docs/qualidade-dados.md`.

**Como executar**
- `Run All` processa as tabelas em sequência. Com `gravar = nao` (padrão), cada tabela é tratada e **validada**, sem gravar nada.
- Com `gravar = sim`, cada tabela só é gravada (Delta, `overwrite`) depois de passar em todas as validações, e a contagem gravada é conferida.
- Uma tabela que falha não é gravada; a seção **Resumo** falha no fim se alguma tabela falhou ou não foi executada.
- Ordem obrigatória: `unidades_consumidoras` antes de `faturas` e `leituras` (checagem de UC fora do cadastro).

**Regras comuns da Silver**
- Tabelas em `v_energy.silver.<mesmo nome da Bronze>`, com os **mesmos nomes de coluna** da Bronze (renomeação só na Gold).
- Datas como `date`/`timestamp`, valores e kWh como `decimal`, IDs como texto.
- Problema que não dá para corrigir vira coluna `flag_<motivo>` (true/false), sem apagar a linha.
- `source_file` e `ingestion_timestamp` da Bronze são mantidas.
- Nenhum valor preenchido pode virar nulo em silêncio: toda conversão é conferida antes de gravar.

**Confidencial:** as saídas das células têm contagens derivadas do dataset. Limpe os outputs antes de exportar ou commitar.

## 0. Configuração
Execute as duas células desta seção antes de qualquer tabela.

In [0]:
import traceback
from datetime import datetime

from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.dropdown("gravar", "nao", ["nao", "sim"], "Gravar na Silver?")
GRAVAR = dbutils.widgets.get("gravar") == "sim"

CATALOG = "v_energy"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
COLS_TECNICAS = ["source_file", "ingestion_timestamp"]
NULOS_DISFARCADOS = ["", "NULL", "N/A", "NA", "-", "NONE", "NAN"]
TABELAS_ESPERADAS = ['unidades_consumidoras', 'faturas']
VALIDACAO_ESPERADA = False  # seção 9 (validação cruzada) faz parte deste notebook

if GRAVAR:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

RESULTADOS = {}  # status de cada tabela, usado no Resumo
SILVER_DF = {}  # DataFrames tratados nesta execução (as tabelas seguintes usam daqui)
print(f"Bronze: {BRONZE} | Silver: {SILVER} | gravar = {'sim' if GRAVAR else 'nao'}")

In [0]:
# ---------- Limpeza e conversão ----------

def limpar(c):
    """Tira espaços das pontas e transforma nulos disfarçados ('', NULL, N/A, -...) em nulo."""
    s = F.trim(F.col(c))
    return F.when(F.upper(s).isin(NULOS_DISFARCADOS), None).otherwise(s)


def limpar_todas(df, colunas):
    return df.select(*[limpar(c).alias(c) if c in colunas else F.col(c) for c in df.columns])


def sem_acento(c):
    """Maiúsculas, sem espaços nas pontas e sem acento: usado para comparar e mapear grafias."""
    return F.translate(F.upper(F.trim(c)), "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")


FORMATOS_DATA = [  # formatos confirmados no profiling (o americano pelos valores acima de 12)
    (r"^\d{4}-\d{2}-\d{2}$", "yyyy-MM-dd"),
    (r"^\d{2}/\d{2}/\d{4}$", "dd/MM/yyyy"),
    (r"^\d{2}\.\d{2}\.\d{4}$", "dd.MM.yyyy"),
    (r"^\d{2}-\d{2}-\d{4}$", "MM-dd-yyyy"),
]


def data_multi(c):
    """Converte cada linha pelo seu formato. Formato desconhecido vira nulo e é barrado por exigir_sem_perda."""
    regra = None
    for padrao, formato in FORMATOS_DATA:
        convertida = F.expr(f"CAST(try_to_timestamp(`{c}`, '{formato}') AS DATE)")
        regra = F.when(F.col(c).rlike(padrao), convertida) if regra is None else regra.when(F.col(c).rlike(padrao), convertida)
    return regra


def dec(c, precisao=12, escala=2):
    """Texto para decimal; vírgula decimal vira ponto. try_cast não derruba a célula no modo ANSI."""
    return F.expr(f"try_cast(replace(`{c}`, ',', '.') AS DECIMAL({precisao},{escala}))")


def ts(c, formato="yyyy-MM-dd HH:mm:ss"):
    return F.expr(f"try_to_timestamp(`{c}`, '{formato}')")


# ---------- Travas: se alguma falhar, a tabela não é gravada ----------

def guardar_texto(df, colunas):
    """Guarda o texto original das colunas que serão convertidas, para conferir depois."""
    for c in colunas:
        df = df.withColumn(f"_txt_{c}", F.col(c))
    return df


def exigir_sem_perda(df, colunas):
    """Nenhum valor preenchido pode virar nulo na conversão (formato inesperado)."""
    r = df.select(*[
        F.sum((F.col(f"_txt_{c}").isNotNull() & F.col(c).isNull()).cast("int")).alias(c) for c in colunas
    ]).first().asDict()
    perdidos = {c: n for c, n in r.items() if n}
    assert not perdidos, f"valores preenchidos que viraram nulo na conversão: {perdidos}"
    return df.drop(*[f"_txt_{c}" for c in colunas])


def exigir_zero(df, regras):
    """regras = {nome: condição}. Toda condição precisa ter 0 linhas."""
    r = df.select(*[
        F.sum(F.coalesce(cond, F.lit(False)).cast("int")).alias(nome) for nome, cond in regras.items()
    ]).first().asDict()
    violadas = {nome: n for nome, n in r.items() if n}
    assert not violadas, f"regras violadas (linhas): {violadas}"


def exigir_chave_unica(df, chave):
    total = df.count()
    unicos = df.select(*chave).distinct().count()
    assert total == unicos, f"chave {chave} duplicada: {total - unicos:,} linhas"
    return total


def exigir_colunas(df, esperadas):
    """A Silver seleciona colunas pelo nome: coluna nova ou ausente na Bronze é erro, nunca descarte silencioso."""
    origem = set(df.columns) - set(COLS_TECNICAS)
    assert origem == set(esperadas), f"colunas diferentes do esperado. Sobrando: {origem - set(esperadas)} | Faltando: {set(esperadas) - origem}"


def exigir_sem_orfaos(df, nome):
    orfaos = df.join(obter("unidades_consumidoras").select("id_uc"), "id_uc", "left_anti").count()
    assert orfaos == 0, f"{nome}: {orfaos:,} linhas com id_uc fora do cadastro"


def contar_flags(df):
    flags = [c for c in df.columns if c.startswith("flag_")]
    if not flags:
        return {}
    return df.select(*[F.sum(F.col(c).cast("int")).alias(c) for c in flags]).first().asDict()


# ---------- Execução ----------

def obter(nome):
    """DataFrame tratado nesta execução; se a seção não rodou agora, lê a tabela Silver já gravada."""
    return SILVER_DF[nome] if nome in SILVER_DF else spark.table(f"{SILVER}.{nome}")


def executar(nome, construir):
    """Trata e valida; só grava com gravar = sim e depois de passar em todas as travas."""
    print(f"=== {nome}")
    try:
        df, info = construir()
        SILVER_DF[nome] = df
        if GRAVAR:
            destino = f"{SILVER}.{nome}"
            df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
            gravadas = spark.table(destino).count()
            assert gravadas == info["linhas"], f"gravadas {gravadas:,} != validadas {info['linhas']:,}"
            SILVER_DF[nome] = spark.table(destino)
            status = "gravada e conferida"
        else:
            status = "validada (sem gravar)"
        RESULTADOS[nome] = {"status": status, **info}
        print(f"[OK] {nome}: {status} | {info}")
    except Exception as e:
        RESULTADOS[nome] = {"status": "FALHA", "erro": f"{type(e).__name__}: {e}"}
        print(f"[FALHA] {nome}: nada foi gravado.")
        traceback.print_exc()

## 1. unidades_consumidoras
`v_energy.bronze.unidades_consumidoras` → `v_energy.silver.unidades_consumidoras`. Regras UC-01 a UC-11 de `docs/qualidade-dados.md`.

**Tratamentos**
- Cabeçalho repetido removido (UC-01); espaços e nulos disfarçados convertidos em nulo (UC-09).
- `nome_titular` em maiúsculas (UC-04); `cpf_cnpj` só com dígitos, como texto (UC-05); `uf` padronizada para `PE`, depois de conferir que todas as grafias são de Pernambuco (UC-06); `classe` com as 5 grafias do log (UC-07).
- Cópias exatas e cópias com formatação diferente removidas depois da padronização (UC-02, UC-03): 1 linha por `id_uc`.
- `data_ligacao` como `date`; `latitude` e `longitude` como `decimal(9,6)` (UC-11: precisão de origem de 2 casas).

**Flag, sem apagar a linha**
- `flag_cep_ausente`: CEP nulo ou sem 8 dígitos; o CEP inválido vira nulo (UC-08).

**Validações antes de gravar:** `id_uc` único, `classe` dentro do mapa, coerência classe × subgrupo, nenhuma conversão perdida.

In [0]:
CLASSES = {"RESIDENCIAL": "Residencial", "COMERCIAL": "Comercial", "INDUSTRIAL": "Industrial",
           "RURAL": "Rural", "PODER PUBLICO": "Poder Público"}
SUBGRUPO_DA_CLASSE = {"Residencial": "B1", "Rural": "B2", "Comercial": "B3", "Poder Público": "B3", "Industrial": "A4"}


def construir_unidades_consumidoras():
    df = spark.table(f"{BRONZE}.unidades_consumidoras")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = df.filter(F.upper(F.trim("id_uc")) != "ID_UC")                       # UC-01
    df = limpar_todas(df, origem)                                              # UC-09
    exigir_zero(df, {"uf_fora_de_pe": F.col("uf").isNotNull() & ~F.upper("uf").isin("PE", "PERNAMBUCO", "P.E.")})

    mapa_classe = F.create_map(*[F.lit(x) for kv in CLASSES.items() for x in kv])
    df = (df
        .withColumn("nome_titular", F.upper("nome_titular"))                   # UC-04
        .withColumn("cpf_cnpj", F.regexp_replace("cpf_cnpj", r"[^0-9]", ""))   # UC-05 (texto)
        .withColumn("uf", F.lit("PE"))                                         # UC-06
        .withColumn("classe", mapa_classe[sem_acento("classe")])               # UC-07
        .withColumn("cep", F.regexp_replace("cep", r"[^0-9]", ""))
        .withColumn("cep", F.when(F.length("cep") == 8, F.col("cep")))         # UC-08: inválido vira nulo
        .withColumn("flag_cep_ausente", F.col("cep").isNull()))

    df = guardar_texto(df, ["data_ligacao", "latitude", "longitude"])
    df = (df
        .withColumn("data_ligacao", data_multi("data_ligacao"))
        .withColumn("latitude", dec("latitude", 9, 6))                          # UC-11
        .withColumn("longitude", dec("longitude", 9, 6)))
    df = exigir_sem_perda(df, ["data_ligacao", "latitude", "longitude"])

    df = df.dropDuplicates(origem)                                             # UC-02 e UC-03
    df = df.select(*origem, "flag_cep_ausente", *COLS_TECNICAS)

    total = exigir_chave_unica(df, ["id_uc"])
    mapa_sub = F.create_map(*[F.lit(x) for kv in SUBGRUPO_DA_CLASSE.items() for x in kv])
    exigir_zero(df, {
        "classe_fora_do_mapa": F.col("classe").isNull(),
        "classe_x_subgrupo": mapa_sub[F.col("classe")] != F.col("subgrupo_tarifario"),
    })
    return df, {"linhas": total, **contar_flags(df)}


executar("unidades_consumidoras", construir_unidades_consumidoras)

## 2. faturas
`v_energy.bronze.faturas` → `v_energy.silver.faturas`. Regras FAT-01 a FAT-05. Depende de `unidades_consumidoras` (seção 1).

**Tratamentos**
- Cabeçalho repetido removido (FAT-01).
- `data_emissao` em 4 formatos convertida linha a linha (FAT-02); `data_vencimento` e `data_pagamento` como `date`.
- `valor_total` no formato `R$ 1.234,56` convertido (FAT-03); valores como `decimal(12,2)`.
- `data_pagamento` com ano `9999` (código de "não pago") vira nulo (FAT-04).

**Coluna derivada:** `status_pagamento` (`PAGA`, `PAGA COM ATRASO`, `EM ABERTO`), comparando `data_pagamento` com `data_vencimento`. O `status` original é mantido.

**Flag, sem apagar a linha**
- `flag_uc_orfa`: fatura de UC que não existe no cadastro (FAT-05). O destino final (quarentena) é decidido na integridade referencial.

**Validações antes de gravar:** `id_fatura` único, bandeira no domínio, `mes_referencia` em `AAAA-MM`, `status` coerente com `data_pagamento`, pagamento nunca antes da emissão, `valor_total = valor_energia + tributos`.

In [0]:
BANDEIRAS = ["Verde", "Amarela", "Vermelha P1", "Vermelha P2"]


def construir_faturas():
    df = spark.table(f"{BRONZE}.faturas")
    origem = [c for c in df.columns if c not in COLS_TECNICAS]

    df = df.filter(F.upper(F.trim("id_fatura")) != "ID_FATURA")               # FAT-01
    df = limpar_todas(df, origem)
    df = df.withColumn("data_pagamento",                                       # FAT-04: 9999 = não pago
        F.when(F.col("data_pagamento").startswith("9999"), None).otherwise(F.col("data_pagamento")))

    convertidas = ["data_emissao", "data_vencimento", "data_pagamento", "valor_energia", "tributos", "valor_total"]
    df = guardar_texto(df, convertidas)
    vt = F.col("valor_total")
    df = (df
        .withColumn("data_emissao", data_multi("data_emissao"))                # FAT-02
        .withColumn("data_vencimento", data_multi("data_vencimento"))
        .withColumn("data_pagamento", data_multi("data_pagamento"))
        .withColumn("valor_total", F.when(vt.contains(","),                    # FAT-03: R$ 1.234,56
            F.regexp_replace(F.regexp_replace(F.regexp_replace(vt, r"R\$\s*", ""), r"\.", ""), ",", ".")).otherwise(vt)))
    for c in ["valor_energia", "tributos", "valor_total"]:
        df = df.withColumn(c, dec(c, 12, 2))
    df = exigir_sem_perda(df, convertidas)

    df = df.withColumn("status_pagamento",
        F.when(F.col("data_pagamento").isNull(), "EM ABERTO")
         .when(F.col("data_pagamento") > F.col("data_vencimento"), "PAGA COM ATRASO")
         .otherwise("PAGA"))

    ucs = obter("unidades_consumidoras").select("id_uc", F.lit(True).alias("_existe"))
    df = (df.join(ucs, "id_uc", "left")                                        # FAT-05
            .withColumn("flag_uc_orfa", F.col("_existe").isNull()).drop("_existe"))
    df = df.select(*origem, "status_pagamento", "flag_uc_orfa", *COLS_TECNICAS)

    total = exigir_chave_unica(df, ["id_fatura"])
    exigir_zero(df, {
        "bandeira_fora_do_dominio": ~F.col("bandeira_tarifaria").isin(BANDEIRAS),
        "mes_referencia_fora_do_formato": ~F.col("mes_referencia").rlike(r"^\d{4}-\d{2}$"),
        "status_x_data_pagamento": (F.col("status") == "EM ABERTO") != (F.col("status_pagamento") == "EM ABERTO"),
        "pagamento_antes_da_emissao": F.col("data_pagamento") < F.col("data_emissao"),
        "total_diferente_de_energia_mais_tributos":
            F.abs(F.col("valor_total") - F.col("valor_energia") - F.col("tributos")) > 0.01,
    })
    return df, {"linhas": total, **contar_flags(df)}


executar("faturas", construir_faturas)

## 3. Resumo
Imprime o estado de cada tabela. Output **confidencial** (contagens reais): nunca no Git.
Falha se alguma tabela não foi executada ou falhou.

In [0]:
print("CONFIDENCIAL: resultados derivados do dataset. Não versionar.")
print(f"gravar = {'sim' if GRAVAR else 'nao'} | {datetime.now().isoformat(timespec='seconds')}")
esperadas = TABELAS_ESPERADAS + (["validacao_cruzada"] if VALIDACAO_ESPERADA else [])
incompletas = []
for nome in esperadas:
    r = RESULTADOS.get(nome)
    if r is None:
        print(f"- {nome}: NÃO EXECUTADA")
        incompletas.append(nome)
        continue
    print(f"- {nome}: {r}")
    if r["status"] == "FALHA":
        incompletas.append(nome)

assert not incompletas, f"Silver NÃO concluída. Não executadas ou com falha: {incompletas}"
print("[OK] Todas as tabelas tratadas e validadas" + (" e gravadas." if GRAVAR else " (nada foi gravado: gravar = nao)."))